# ReviewSense PT-BR — 02 · Baseline clássico

Fase 2 em **CPU local**: DummyClassifier e TF-IDF + LinearSVC. Apenas treino e
validação congelados da Fase 1. O CSV de teste final não é aberto.

O notebook reutiliza os módulos do pacote. Por padrão **reabre uma execução salva**,
sem repetir a busca. Consulte `docs/FASE2_BASELINE.md` para decisões, funções e protocolo.
O corpus B2W-Reviews01 mantém a licença original CC BY-NC-SA 4.0; o modelo e seu
vocabulário ficam locais, fora do Git. Somente agregados são exibidos aqui.

## 1. Ambiente e configuração

Na raiz, instale `python -m pip install -r requirements-dev.txt` e selecione o kernel
Python da `.venv`. Não é preciso configurar GPU, TPU ou Colab.
Defina `REVIEWSENSE_REPO` se o notebook estiver fora da pasta do repositório.

In [ ]:
import json
import os
from pathlib import Path

candidates = [Path(os.environ["REVIEWSENSE_REPO"])] if "REVIEWSENSE_REPO" in os.environ else [
    Path.cwd(), Path.cwd().parent
]
ROOT = next((path.resolve() for path in candidates
             if (path / "src/reviewsense/baseline.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Defina REVIEWSENSE_REPO com a raiz do repositório.")
os.chdir(ROOT)

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from reviewsense.baseline import candidate_parameters, load_experiment, run_experiment
from reviewsense.evaluate import classification_metrics, plot_confusion_matrix
from reviewsense.frozen_data import load_training_data

ARTIFACTS = ROOT / "data/processed/phase1-final"
REFERENCE = ROOT / "reports/data_audit_summary.json"
RUN_DIR = ROOT / "models/baseline/phase2-v1"
config = json.loads((ROOT / "configs/baseline.json").read_text(encoding="utf-8"))
display(pd.DataFrame(candidate_parameters(config)))

## 2. Verificação dos splits congelados

`load_training_data` abre **somente train.csv e val.csv**. Compara conteúdo, IDs,
contagens e fingerprints com o manifest local e com o resumo versionado da Fase 1.
Não recria, embaralha ou altera splits; não usa a função que carrega o teste.

In [ ]:
train, val, provenance = load_training_data(ARTIFACTS, REFERENCE)
display(pd.DataFrame(provenance["splits"]).T[["rows", "fingerprint"]])
assert provenance["test_csv_opened"] is False

## 3. Treinar uma vez ou reabrir a execução

Execute no terminal, na raiz:

```bash
.venv/bin/python -m reviewsense.baseline
```

Alternativamente, altere `TRAIN = True` abaixo. O diretório de saída deve estar
ausente; use outro `RUN_DIR` para uma nova execução. O treino usa somente treino;
a escolha é pelo F1 macro de validação. Não há refit em treino+validação.
Os artefatos joblib devem ser de origem local confiável.

In [ ]:
TRAIN = False
if TRAIN:
    run_experiment(ARTIFACTS, REFERENCE, config, RUN_DIR)
if not (RUN_DIR / "results.json").exists():
    raise FileNotFoundError("Execute o treinamento pelo CLI ou ative TRAIN em uma saída nova.")
model, results = load_experiment(RUN_DIR)
assert results["data"]["splits"] == provenance["splits"]
assert results["data"]["test_csv_opened"] is False
selected_metrics = classification_metrics(val["label"], model.predict(val["review_text"].tolist()))
assert selected_metrics == results["selected"]["metrics"]
print("Execução salva verificada; nenhuma busca repetida no modo padrão.")

## 4. Busca e seleção

- `C`: 0,1 / 1 / 10, do maior para o menor nível de regularização.
- N-grams: palavras isoladas ou palavras e pares, incluindo expressões como “não gostei”.
- `class_weight`: padrão ou balanceado, para testar a influência do desbalanceamento.
- Até 12 candidatos sequenciais; vocabulário limitado a 80 mil features.
- Empate: primeiro candidato na ordem da grade; empate global favorece Dummy.
- Candidatos que não convergirem não podem ser selecionados.

TF-IDF representa frequência e raridade dos termos; LinearSVC aprende separações
lineares um-contra-restante. Dummy prevê sempre a classe mais frequente do treino.

In [ ]:
search_table = pd.DataFrame([
    {"candidato": row["candidate_id"], **row["params"],
     "f1_macro": row["metrics"]["f1_macro"],
     "precision_macro": row["metrics"]["precision_macro"],
     "recall_macro": row["metrics"]["recall_macro"],
     "treino_s": row["fit_seconds"], "convergiu": row["converged"]}
    for row in results["candidates"]
])
display(search_table)
best = next(row for row in results["candidates"]
            if row["candidate_id"] == results["best_svm_candidate_id"])
comparison = {"Dummy": results["dummy"], "Melhor SVM": best}
display(pd.DataFrame({
    name: {key: row["metrics"][key]
           for key in ["f1_macro", "precision_macro", "recall_macro", "accuracy"]}
    for name, row in comparison.items()
}).T)
print("Selecionado:", results["selected"]["name"], results["selected"]["candidate_id"])

In [ ]:
for name, record in comparison.items():
    print(name)
    display(pd.DataFrame(record["metrics"]["per_class"]).T)
    figure = plot_confusion_matrix(record["metrics"], title=f"Validação — {name}")
    plt.show()
    plt.close(figure)

## 5. Tempos e condições

Treino inclui ajuste do TF-IDF e SVM para cada candidato. Inferência inclui
transformação do texto e predição, sem leitura de disco. São três passagens de toda
a validação, lotes de 256, após um lote de aquecimento. O custo por texto é uma
média amortizada em lote; não é latência de uma requisição individual.
Tempos variam com carga, temperatura e hardware. Não comparam CPU com GPU.

In [ ]:
display(pd.Series({key: results["environment"][key]
                   for key in ["cpu_model", "logical_cpus", "ram_bytes", "device", "python"]}))
display(pd.Series(results["environment"]["versions"], name="versão"))
display(pd.DataFrame({
    name: {"treino_s": row["fit_seconds"],
           "inferencia_validacao_s_mediana": row["inference"]["median_seconds"],
           "ms_por_texto_em_lote": row["inference"]["milliseconds_per_text_amortized"]}
    for name, row in comparison.items()
}).T)
display(pd.Series(results["timing"]))

## Limitações e próximo passo

Esta validação foi usada para selecionar hiperparâmetros; não é uma estimativa
final independente. A classe neutra pode conter sentimentos mistos e as estrelas
são rótulos indiretos. O TF-IDF de palavras perde boa parte da pontuação, emojis
e contexto longo; preserva acentos e não remove stopwords. O modelo retorna classes,
sem probabilidades calibradas.

Nenhum exemplo, ID individual ou termo do vocabulário é publicado nas saídas.
O teste final permanece reservado. BERTimbau e GPU ficam fora desta fase.